# Stage 1: LoRA Supervised Fine-Tuning (SFT) on Mistral 7B

**Environment:** Kaggle GPU (P100/T4)  
**Estimated time:** 6–8 hours  
**Output:** LoRA adapter weights in `models/sft_adapter/`

This notebook fine-tunes Mistral 7B Instruct using QLoRA (4-bit NF4) on
domain-specific instruction data extracted from the preference dataset's
`chosen` responses. The LoRA adapter produced here is the input for Stage 2 (DPO).

## 1. Install Dependencies

Kaggle pre-installs PyTorch but we need the latest HuggingFace stack.

In [1]:
import os
for root, dirs, files in os.walk('/kaggle/input/'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/aadityae/hr-preference-data/train.json
/kaggle/input/datasets/aadityae/hr-preference-data/test.json


In [2]:
!pip install -q \
    transformers>=4.36.0 \
    trl>=0.7.0 \
    peft>=0.7.0 \
    bitsandbytes>=0.41.0 \
    accelerate>=0.25.0 \
    wandb>=0.16.0 \
    datasets>=2.16.0

## 2. Imports & Configuration

In [3]:
import os
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# ── Hyperparameters (from src/training/config.py) ──
BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.2"

# Quantization
LOAD_IN_4BIT = True
BNB_4BIT_QUANT_TYPE = "nf4"
BNB_4BIT_COMPUTE_DTYPE = "float16"
BNB_4BIT_USE_DOUBLE_QUANT = True

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]

# Training
SFT_EPOCHS = 3
SFT_BATCH_SIZE = 4
SFT_GRADIENT_ACCUMULATION_STEPS = 4
SFT_LEARNING_RATE = 2e-4
SFT_MAX_SEQ_LENGTH = 1024
SFT_WARMUP_RATIO = 0.03
SFT_SAVE_STEPS = 500
SFT_LOGGING_STEPS = 25

# Paths
SFT_OUTPUT_DIR = "./sft_output"
SFT_ADAPTER_DIR = "/kaggle/working/sft_adapter"
DATASET_PATH = "/kaggle/input/datasets/aadityae/hr-preference-data/train.json"

# Resume training (set to a checkpoint path to resume, or None)
RESUME_FROM_CHECKPOINT = None  # e.g. "./sft_output/checkpoint-500"

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: Tesla P100-PCIE-16GB
Memory: 17.1 GB


## 3. Weights & Biases Setup

Set your W&B API key as a Kaggle secret named `WANDB_API_KEY`, or paste it below.

In [4]:
import wandb

# Try Kaggle secrets first, fall back to manual entry
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    wandb_key = secrets.get_secret("WANDB_API_KEY")
except Exception:
    wandb_key = os.environ.get("WANDB_API_KEY", "")

if wandb_key:
    wandb.login(key=wandb_key)
    os.environ["WANDB_PROJECT"] = "air-india-rlhf"
    os.environ["WANDB_LOG_MODEL"] = "false"
    print("W&B logged in.")
else:
    os.environ["WANDB_DISABLED"] = "true"
    print("W&B disabled — no API key found.")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aaditya4401 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B logged in.


## 4. Load & Prepare Dataset

We use the `chosen` responses from the preference dataset as SFT targets.
Each example is formatted as a Mistral instruction:
```
[INST] {prompt} [/INST] {chosen_response}
```

In [5]:
raw_ds = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Loaded {len(raw_ds)} examples")
print(raw_ds[0].keys())


def format_instruction(example):
    """Format as Mistral instruct template."""
    text = f"[INST] {example['prompt']} [/INST] {example['chosen']}"
    return {"text": text}


sft_ds = raw_ds.map(format_instruction)
print(f"\nSample:\n{sft_ds[0]['text'][:300]}...")

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 648 examples
dict_keys(['prompt', 'chosen', 'rejected'])


Map:   0%|          | 0/648 [00:00<?, ? examples/s]


Sample:
[INST] What expenses are reimbursable during business travel? [/INST] During business travel, the following expenses are reimbursable:

- **Meals**: 
  - Company-provided meals while traveling.
  - Up to $100 USD (or local equivalent) per day for meals.
  - Tips are acceptable up to 20% (cash tips u...


## 5. Load Base Model with 4-bit Quantization

Using bitsandbytes NF4 quantization to fit Mistral 7B in ~5 GB VRAM.
Double quantization further reduces the memory footprint.

In [6]:
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=LOAD_IN_4BIT,
#     bnb_4bit_quant_type=BNB_4BIT_QUANT_TYPE,
#     bnb_4bit_compute_dtype=getattr(torch, BNB_4BIT_COMPUTE_DTYPE),
#     bnb_4bit_use_double_quant=BNB_4BIT_USE_DOUBLE_QUANT,
# )
bnb_config = BitsAndBytesConfig(
    load_in_4bit=LOAD_IN_4BIT,
    bnb_4bit_quant_type=BNB_4BIT_QUANT_TYPE,
    bnb_4bit_compute_dtype=torch.float16,  # make sure this is float16, not bfloat16
    bnb_4bit_use_double_quant=BNB_4BIT_USE_DOUBLE_QUANT,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Prepare model for QLoRA training
model = prepare_model_for_kbit_training(model)

print(f"Model loaded: {BASE_MODEL}")
print(f"Model memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Model loaded: mistralai/Mistral-7B-Instruct-v0.2
Model memory footprint: 4.54 GB


## 6. Configure LoRA Adapters

LoRA injects trainable low-rank matrices into the attention layers.
Only ~0.5% of parameters are trained, the rest stay frozen.

In [7]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879


## 7. Training

Using TRL's `SFTTrainer` which handles chat formatting and packing.
Checkpoints save every 500 steps so we can resume if Kaggle times out.
Gradient checkpointing trades compute for memory to fit on 16 GB GPUs.

In [8]:
print(sft_ds.column_names)
print(sft_ds[0])

['prompt', 'chosen', 'rejected', 'text']
{'prompt': 'What expenses are reimbursable during business travel?', 'chosen': "During business travel, the following expenses are reimbursable:\n\n- **Meals**: \n  - Company-provided meals while traveling.\n  - Up to $100 USD (or local equivalent) per day for meals.\n  - Tips are acceptable up to 20% (cash tips under $50 do not require a receipt).\n\n- **Lodging**: \n  - Expenses related to accommodation during business travel.\n\n- **Transportation**: \n  - Costs associated with travel to and from the business destination.\n\n- **Other allowable expenses**: \n  - Any other expenses deemed “ordinary and necessary” for business purposes.\n\n**Important Notes**:\n- Receipts must be submitted for reimbursement.\n- Team Members should provide a description of the trip/reason for travel.\n- Expenses must comply with the company's guidelines to avoid being flagged as Out of Policy.", 'rejected': 'During business travel, the following expenses are rei

In [9]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [10]:
sft_ds = sft_ds.rename_column("chosen", "completion")
sft_ds = sft_ds.remove_columns(["rejected", "text"])

In [11]:
training_args = TrainingArguments(
    output_dir=SFT_OUTPUT_DIR,
    num_train_epochs=SFT_EPOCHS,
    per_device_train_batch_size=SFT_BATCH_SIZE,
    gradient_accumulation_steps=SFT_GRADIENT_ACCUMULATION_STEPS,
    learning_rate=SFT_LEARNING_RATE,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    logging_steps=SFT_LOGGING_STEPS,
    save_steps=SFT_SAVE_STEPS,
    save_total_limit=3,
    report_to="none",
    run_name="sft-lora-mistral7b",
    optim="paged_adamw_8bit",
)



trainer = SFTTrainer(
    model=model,
    train_dataset=sft_ds,
    args=training_args,
    processing_class=tokenizer,
)
# trainer = SFTTrainer(
#     model=model,
#     train_dataset=sft_ds,
#     args=training_args,
#     processing_class=tokenizer,
#     dataset_text_field="text",
#     packing=False,
# )

print(f"Training samples: {len(sft_ds)}")
print(f"Effective batch size: {SFT_BATCH_SIZE * SFT_GRADIENT_ACCUMULATION_STEPS}")
print(f"Epochs: {SFT_EPOCHS}")
print(f"Checkpoints every {SFT_SAVE_STEPS} steps")
if RESUME_FROM_CHECKPOINT:
    print(f"Resuming from: {RESUME_FROM_CHECKPOINT}")

Adding EOS to train dataset:   0%|          | 0/648 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/648 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/648 [00:00<?, ? examples/s]

Training samples: 648
Effective batch size: 16
Epochs: 3
Checkpoints every 500 steps


In [12]:
# Start (or resume) training
trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
25,1.532943
50,0.820333
75,0.333059
100,0.136645


TrainOutput(global_step=123, training_loss=0.5906733865660381, metrics={'train_runtime': 4043.9506, 'train_samples_per_second': 0.481, 'train_steps_per_second': 0.03, 'total_flos': 2.6644109597835264e+16, 'train_loss': 0.5906733865660381})

## 8. Save LoRA Adapter

Save the trained LoRA weights. These are small (~50–100 MB) compared to
the full 14 GB model, so they're easy to download and move to Stage 2.

In [13]:
os.makedirs(SFT_ADAPTER_DIR, exist_ok=True)

# Save adapter weights + tokenizer
trainer.model.save_pretrained(SFT_ADAPTER_DIR)
tokenizer.save_pretrained(SFT_ADAPTER_DIR)

print(f"SFT adapter saved to {SFT_ADAPTER_DIR}")

# List saved files
for f in os.listdir(SFT_ADAPTER_DIR):
    size = os.path.getsize(os.path.join(SFT_ADAPTER_DIR, f))
    print(f"  {f}: {size / 1e6:.1f} MB")

SFT adapter saved to /kaggle/working/sft_adapter
  tokenizer.json: 3.5 MB
  tokenizer_config.json: 0.0 MB
  adapter_model.safetensors: 27.3 MB
  chat_template.jinja: 0.0 MB
  README.md: 0.0 MB
  adapter_config.json: 0.0 MB


## 9. (Optional) Push to Hugging Face Hub

As a backup in case the Kaggle session expires before you can download.
Set `HF_HUB_REPO` to your repo name (e.g. `your-username/air-india-sft-adapter`).

In [14]:
HF_HUB_REPO = ""  # Set your repo name here

if HF_HUB_REPO:
    # Login — set HF_TOKEN as a Kaggle secret
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        hf_token = secrets.get_secret("HF_TOKEN")
    except Exception:
        hf_token = os.environ.get("HF_TOKEN", "")

    if hf_token:
        trainer.model.push_to_hub(HF_HUB_REPO, token=hf_token)
        tokenizer.push_to_hub(HF_HUB_REPO, token=hf_token)
        print(f"Pushed to https://huggingface.co/{HF_HUB_REPO}")
    else:
        print("No HF_TOKEN found. Skipping Hub push.")
else:
    print("HF_HUB_REPO not set. Skipping Hub push.")

HF_HUB_REPO not set. Skipping Hub push.


## 10. Quick Sanity Check

Generate a sample response from the SFT-tuned model to verify it works.

In [15]:
# Quick inference test
test_prompt = "[INST] What aircraft does Air India use for domestic routes? [/INST]"
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in MistralDecoderLayer. Setting `past_key_values=None`.


[INST] What aircraft does Air India use for domestic routes? [/INST] Air. Q. Q. Question: Question: Q. Question: Question  Q. QQ. Q. Question: Q. Question: Q. Q. Question: Question: Question: Question: Q. Question  Q. Q. Q. Q. Q. Q. Question  Question  Question: Q. Question: Q. Question  Q. Question  Q. Q. Question: Q Q. Q. Question: Q. Question: Question: Q. Question: Question  Question  Question: Q. Question: Q. Q. Q. Question: Question: Question  Question: Question: Question: QUESTS. Q. Question: Q. Que. Q. Q. Q. Question: Question  Question: Question: Question: Q. QS. Question: Q. Question: Q. Q. Question  Question: Q. Q. Question: Q. Question: Question: Question: Q: Q. Question  Question: QUE DE DE


In [16]:
# Finish W&B run
if not os.environ.get("WANDB_DISABLED"):
    wandb.finish()